# Observing System Experiment (OSE)

In auroral imager context, an OSE simulates what a space-based imager would see from a low Earth orbit (LEO) platform. It essentially resamples the ASI mosaics to a set time cadence, pixel resolution, and field of view.

In [ ]:
# import dataclasses
# import copy
# from datetime import datetime
import pathlib
import warnings
# import string
# from typing import Tuple, List, Union
# from collections import namedtuple
# import dateutil.parser

import itertools
from collections import namedtuple
from datetime import datetime

import cartopy.crs
import cartopy.feature as cfeature
from asilib.mission import example_satellite
import asilib

try:
    import fontawesome
    fontawesome_imported = True
except ImportError:
    fontawesome_imported = False
    
import matplotlib.font_manager
import matplotlib.textpath
import matplotlib.path
import matplotlib.pyplot as plt
# import matplotlib.colors
# import matplotlib.dates
# import matplotlib.patches
import numpy as np
import matplotlib.gridspec as gridspec

# import scipy.interpolate
# from scipy.spatial import cKDTree
# import pandas as pd
# import asilib
# import asilib.map
# import asilib.asi
# import cartopy.crs as ccrs
# import pymap3d.los

If you need a nicer marker for the satellite you'll need to install fontawesome and donwload the Font Awesome .otf file (free tier) from their [website](https://fontawesome.com/download). Unzip it and set the fontawesome_path to that directory.

In [ ]:
fontawesome_path = pathlib.Path(__file__).parent / "Font Awesome 7 Free-Solid-900.otf"
warned = False

def getmarker(mID):
    if not fontawesome_imported:
        return "s"
    elif not fontawesome_path.exists():
        if not warned:
            warnings.warn(
                f"A fontawesome font file not found at {fontawesome_path}. Using default marker. "
                f"Download them from https://fontawesome.com/download, unzip the archive, and set "
                f"the fontawesome_path to the font file.")
            warned = True
        return "s"
        
    symbol = fontawesome.icons[mID]
    fp = matplotlib.font_manager.FontProperties(fname=fontawesome_path)

    v, codes = matplotlib.textpath.TextToPath().get_text_path(fp, symbol)
    v = np.array(v)
    mean = np.mean([np.max(v,axis=0), np.min(v, axis=0)], axis=0)
    return matplotlib.path.Path(v-mean, codes, closed=False)

# CINEMA constellation OSE of a turning streamer

In [ ]:
time_range = (datetime(2008, 2, 4, 10, 35), datetime(2008, 2, 4, 10, 55))
aurora_alt = 110

location_codes = [
    'FYKN',
    'INUV',
    'FSIM',
    'WHIT',
    'KIAN',
    ]

asis = asilib.Imagers(
    [asilib.asi.themis(code, time_range=time_range, alt=aurora_alt) for code in location_codes]
    )

# Create the CINEMA constellation ephemeris.
orbit_parameter_tuple_type = namedtuple(
    'orbit_parameter_tuple_type', 
    ['mean_anomaly_deg', 'ltan_hours', 'alt_km']
    )
in_track_separation_minutes = 5
orbit_period_minutes = 95
mean_anomaly_deg = 35
sat_alt = 600
delta_mean_anomaly_deg = 360*in_track_separation_minutes/orbit_period_minutes

center_ltan = 2.25
ltan_hours = [center_ltan-1, center_ltan, center_ltan+1]
mean_anomalies = [
    mean_anomaly_deg+delta_mean_anomaly_deg, 
    mean_anomaly_deg, 
    mean_anomaly_deg-delta_mean_anomaly_deg
    ]
constellation = {
    i:orbit_parameter_tuple_type(
        mean_anomaly_deg=mean_anomaly, 
        ltan_hours=ltan,
        alt_km=sat_alt,
        ) for i, (mean_anomaly, ltan) in enumerate(itertools.product(mean_anomalies, ltan_hours))
    }

ephemeris = [None, None]
for key, value in constellation.items():
    ephemeris_obj = example_satellite.Example_Satellite(
        cadence_s=0.5,
        time_range=time_range,
        mean_anomaly_deg=value.mean_anomaly_deg,
        ltan_hours=value.ltan_hours,
        altitude_km=value.alt_km,
    )
    sat_ephemeris = ephemeris_obj.ephemeris()
    if ephemeris[0] is None:
        ephemeris[0] = sat_ephemeris[0]
        ephemeris[1] = sat_ephemeris[1].reshape(*sat_ephemeris[1].shape, 1)
    else:
        ephemeris[1] = np.concatenate(
            (ephemeris[1], sat_ephemeris[1].reshape(*sat_ephemeris[1].shape, 1)), axis=2
            )

ose = asilib.ose.OSE(
    asis, 
    ephemeris, 
    fov=(55, 65), 
    pixel_resolution=(124, 124),
    roll=7,
    )

fig = plt.figure(figsize=(4, 7.5))
gs = gridspec.GridSpec(nrows=4, ncols=3, figure=fig, height_ratios=(3, 1, 1, 1))

center = (
    np.mean(asis.lon_bounds), np.mean(asis.lat_bounds)
)
projection = cartopy.crs.Orthographic(
    central_longitude=center[0], 
    central_latitude=center[1]
)

ax = fig.add_subplot(gs[0, :], projection=projection)
ax.add_feature(cfeature.LAND, color='grey')
ax.add_feature(cfeature.OCEAN, color='cyan')
ax.add_feature(cfeature.COASTLINE, edgecolor='k')
ax.gridlines(linestyle=':')
ax.set_global()
ax.set_extent(
    (center[0]-20, center[0]+20, center[1]-10, center[1]+8), 
    crs=cartopy.crs.PlateCarree()
    )

bx = np.nan*np.zeros((3, 3), dtype=object)
for i in range(3):
    for j in range(3):
        bx[i, j] = fig.add_subplot(gs[i+1, j])
        bx[i, j].set_aspect('equal')
        bx[i, j].xaxis.set_visible(False)
        bx[i, j].yaxis.set_visible(False)
plt.suptitle(
    f'CINEMA OSE | fov={ose.fov} [deg]\n'
    f'alt={sat_alt} [km] | resolution={ose.pixel_resolution} [px]', 
    fontsize=12
    )
plt.subplots_adjust(
    bottom=0.01, top=0.95, left=0.01, right=0.99, wspace=0.03, hspace=0.03
)
save_name = (
    f'{time_range[0].strftime("%Y%m%d_%H%M%S")}_{time_range[-1].strftime("%H%M%S")}'
    f'_cinema_ose_{sat_alt=}km_ltan{round(ltan_hours[1])}_{aurora_alt=}km.mp4'
    )
ose.animate_ose(ax=ax, bx=bx, animation_name=save_name, pcolormesh_kwargs={'rasterized':True})